In [1]:
from jaxcmr.memorysearch import (
    BaseCMR,
    InstanceCMR,
    uniform_presentations_data_likelihood,
    variable_presentations_data_likelihood,
    predict_and_simulate_trial,
    predict_and_simulate_pres_and_trial,
    log_likelihood,
    start_retrieving,
    experience,
    trial_item_count,
    trial_list_length,
)
from jax import numpy as jnp, jit, lax, vmap, disable_jit

In [2]:
parameters = {
        "encoding_drift_rate": 0.8016327576683261,
        "delay_drift_rate": 0.9966411723460118,
        "start_drift_rate": 0.051123130268380085,
        "recall_drift_rate": 0.8666706252504806,
        "shared_support": 0.016122091797498662,
        "item_support": 0.8877852952105489,
        "learning_rate": 0.10455606050373444,
        "primacy_scale": 33.57091895097917,
        "primacy_decay": 1.57091895097917,
        "stop_probability_scale": 0.0034489993376706257,
        "stop_probability_growth": 0.3779780110633191,
        "choice_sensitivity": 1.0,
        "mcf_trace_sensitivity": 1.0,
    }

item_count = 10
max_presentation_count = 10
presentation = jnp.array(
    [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [1, 2, 3, 4, 5, 6, 7, 8, 9, 9]]
)
trial = jnp.array([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [6]:
item_counts = vmap(trial_item_count)(presentation)
item_counts

Array([10,  9], dtype=int32)

In [12]:
def f1():
    return predict_and_simulate_pres_and_trial(
        InstanceCMR.create,
        10,
        presentation[0],
        trial[0],
        parameters,
    )[1]

def f2():
    return predict_and_simulate_pres_and_trial(
        InstanceCMR.create,
        9,
        presentation[1],
        trial[1],
        parameters,
    )[1]

@jit
def f3():
    return jnp.array([f1(), f2()])

f3()

Array([[0.6772254 , 0.00503323, 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ],
       [0.6563652 , 0.00503323, 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ]],      dtype=float32)

In [13]:
functions = [f1, f2]

@jit
def f4():
    return jnp.array([f() for f in functions])

f4()

Array([[0.6772254 , 0.00503323, 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ],
       [0.6563652 , 0.00503323, 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ]],      dtype=float32)

In [18]:
from jax.tree_util import Partial

f1 = Partial(predict_and_simulate_pres_and_trial, InstanceCMR.create, 10)
f2 = Partial(predict_and_simulate_pres_and_trial, InstanceCMR.create, 9)

f1(presentation[0], trial[0], parameters)[1]

Array([0.6772254 , 0.00503323, 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ],      dtype=float32, weak_type=True)

In [22]:
functions = [
    Partial(predict_and_simulate_pres_and_trial, InstanceCMR.create, item_count) 
    for item_count in item_counts
    ]

@jit
def f4(trials, presentations):
    trial_indices = jnp.arange(trials.shape[0])
    return lax.map(lambda trial_index: functions[trial_index](trials[trial_index], presentations[trial_index]))

f4(trial, presentation)

ValueError: Non-hashable static arguments are not supported, as this can lead to unexpected cache-misses. Static argument (index 1) of type <class 'jaxlib.xla_extension.ArrayImpl'> for function predict_and_simulate_pres_and_trial is non-hashable.